# Решения: RFM через groupby

**Для преподавателя.** Секционный эталон урока и ДЗ; до сдачи ученикам не показывать.


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (Path(name), Path("../../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в ../../data")

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## Урок. 1. Проверка ключей

In [ ]:
key_checks={"orders":orders["order_id"].is_unique,"customers":customers["customer_id"].is_unique,"payments":payments["order_id"].is_unique}
assert set(key_checks.values())=={True}


## Урок. 2. Таблица заказ+оплата

In [ ]:
merged=orders.merge(payments,on="order_id",validate="one_to_one")
assert len(merged)==3500


## Урок. 3. Опорная дата

In [ ]:
ref_date=orders["order_purchase_timestamp"].max()
REF_NOTE="Одна общая опорная дата делает Recency сопоставимым между клиентами. Если взять максимум внутри каждого клиента, Recency станет нулём для всех и потеряет смысл давности."
assert len(REF_NOTE)>=160


## Урок. 4. Frequency

In [ ]:
frequency=merged.groupby("customer_id")["order_id"].nunique()
assert int(frequency.sum())==3500


## Урок. 5. Monetary

In [ ]:
monetary=merged.groupby("customer_id")["payment_value"].sum()
assert np.isclose(monetary.sum(),payments["payment_value"].sum())


## Урок. 6. Recency

In [ ]:
last_purchase=merged.groupby("customer_id")["order_purchase_timestamp"].max()
recency=(ref_date-last_purchase).dt.days
assert recency.min()==0


## Урок. 7. Сборка RFM

In [ ]:
rfm=pd.concat([recency.rename("Recency"),frequency.rename("Frequency"),monetary.rename("Monetary")],axis=1).reset_index()
assert len(rfm)==778


## Урок. 8. Профиль и интерпретация

In [ ]:
stats=rfm[["Recency","Frequency","Monetary"]].describe(); top5=rfm.nlargest(5,"Monetary")
RFM_NOTE="Высокий Monetary при низком Frequency означает редкие крупные покупки, а не высокую лояльность автоматически. Recency показывает, насколько давно была последняя покупка. CRM должна читать три измерения вместе и проверить устойчивость сегмента, прежде чем выбирать коммуникацию."
assert len(RFM_NOTE)>=220


## ДЗ. 1. Чистая функция build_rfm

In [ ]:
def build_rfm(orders_df,payments_df):
    merged=orders_df.merge(payments_df,on="order_id",validate="one_to_one")
    ref_date=merged["order_purchase_timestamp"].max()
    out=merged.groupby("customer_id").agg(last=("order_purchase_timestamp","max"),Frequency=("order_id","nunique"),Monetary=("payment_value","sum")).reset_index()
    out["Recency"]=(ref_date-out["last"]).dt.days
    return out[["customer_id","Recency","Frequency","Monetary"]]
rfm=build_rfm(orders,payments)


## ДЗ. 2. Frequency bands

In [ ]:
rfm["freq_band"]=pd.cut(rfm["Frequency"],[0,2,5,np.inf],labels=["1-2","3-5","6+"])
freq_share=rfm["freq_band"].value_counts(normalize=True,sort=False)
assert np.isclose(freq_share.sum(),1)


## ДЗ. 3. Давние клиенты

In [ ]:
oldest=rfm.nlargest(10,"Recency")
assert oldest["Recency"].is_monotonic_decreasing


## ДЗ. 4. Challenge: сверка инвариантов

In [ ]:
checks={"customers":len(rfm)==customers["customer_id"].nunique(),"orders":int(rfm["Frequency"].sum())==len(orders),"money":np.isclose(rfm["Monetary"].sum(),payments["payment_value"].sum()),"nonnegative_recency":rfm["Recency"].ge(0).all()}
assert set(checks.values())=={True}


## ДЗ. 5. Challenge: CRM-рекомендация

In [ ]:
CRM_NOTE="Recency помогает выделить давно не покупавших, Frequency — регулярных, Monetary — клиентов с высокой суммой покупок. CRM может начать с персональной проверки редких дорогих клиентов и отдельной коммуникации частым. Ограничение: RFM описывает историю, не доказывает будущую покупку и не является меткой churn."
assert len(CRM_NOTE)>=260
